In [29]:
import os
import math
import pandas as pd
import numpy as np
print(os.getcwd())

/home/yli/MR_CDFT/MRCDFT


In [ ]:
para_path = 'examples/22Ne/22Ne_para.dat'
b23_path = 'examples/22Ne/22Ne_b23.dat'
output_dir = 'examples/22Ne/output'
A=22

# para_path = 'examples/20Ne/para.dat'
# b23_path = 'examples/20Ne/b23.dat'
# output_dir = 'examples/20Ne/output'
# A=20

In [31]:
# extract eMax, nbeta, nphi from parameter file
with open(para_path, "r", encoding="utf-8") as f:
    for line in f:
        line_nocomment = line.split("!")[0]

        if "n0f" in line_nocomment:
            eMax = int(line_nocomment.split("=")[1].split()[0])
        if "nphi" in line_nocomment:
            nphi = int(line_nocomment.split("=")[1])
        if "nbeta" in line_nocomment:
            nbeta = int(line_nocomment.split("=")[1])
print('eMax=',eMax,' nphi=',nphi,' nbeta=',nbeta)

eMax= 6  nphi= 7  nbeta= 12


In [32]:
# Prepare result dataframe, add TD density filename and Kernel filenames

Res_data = pd.DataFrame(columns=['beta2(qf)','beta3(qf)', 'beta2(qi)','beta3(qi)', 'TDdens_filename', 'Kernel_filename'])

b23_data = pd.read_csv(b23_path,
                    sep=r'\s+',   # 按空格分隔
                    skiprows=0)
for q1 in range(len(b23_data)):
    for q2 in range(len(b23_data)):

        beta2q1 = b23_data.iloc[q1]['beta2']
        beta3q1 = b23_data.iloc[q1]['beta3']
        beta2q2 = b23_data.iloc[q2]['beta2']
        beta3q2 = b23_data.iloc[q2]['beta3']

        TDdens_filename = f"Proj_TD1B.1D_eMax{eMax:02d}.{nphi:02d}.{nbeta:02d}{int(beta2q1*100):+04d}{int(beta3q1*100):+04d}_{int(beta2q2*100):+04d}{int(beta3q2*100):+04d}.me"
        if not os.path.exists(os.path.join(output_dir,TDdens_filename)):
            print(TDdens_filename," not exist !")
        if(q1 <= q2):
            Kernel_filename = f"Proj_kern.1D_eMax{eMax:02d}.{nphi:02d}.{nbeta:02d}{int(beta2q1*100):+04d}{int(beta3q1*100):+04d}_{int(beta2q2*100):+04d}{int(beta3q2*100):+04d}.elem"
        else:
            Kernel_filename = f"Proj_kern.1D_eMax{eMax:02d}.{nphi:02d}.{nbeta:02d}{int(beta2q2*100):+04d}{int(beta3q2*100):+04d}_{int(beta2q1*100):+04d}{int(beta3q1*100):+04d}.elem"
        if not os.path.exists(os.path.join(output_dir,Kernel_filename)):
            print(Kernel_filename," not exist !")
        
        Res_data.loc[len(Res_data)] = [beta2q1, beta3q1, beta2q2, beta3q2, TDdens_filename, Kernel_filename]
Res_data

,beta2(qf),beta3(qf),beta2(qi),beta3(qi),TDdens_filename,Kernel_filename
0,0.0,0.0,0.0,0.0,Proj_TD1B.1D_eMax06.07.12+000+000_+000+000.me,Proj_kern.1D_eMax06.07.12+000+000_+000+000.elem


In [33]:
# read reduced single-particle matrix element of the electromagnetic operator
EM_path = os.path.join(output_dir,f'EM_A{A}_eMax{eMax:02d}.me')

def find_target_line(filename, target_string):
    """
    Find line numbers containing a specific string in a file.
    
    Args:
        filename (str): Name of the file to search
        target_string (str): String to search for
    
    Returns:
        list: List of line numbers containing the target string
    """
    line_numbers = []
    try:
        with open(filename, 'r', encoding='utf-8') as file:
            for line_num, line in enumerate(file, 1):
                if target_string in line:
                    line_numbers.append(line_num)
        return line_numbers
    except FileNotFoundError:
        print(f"File {filename} not found")
        return []
    except Exception as e:
        print(f"Error reading file: {e}")
        return []
nlj_skip_line_number = find_target_line(EM_path, "n l j of a/b")[0] + 1
REM_skip_line_number = find_target_line(EM_path, "reduced single-particle matrix element of the electromagnetic operator")[0] + 1

print(nlj_skip_line_number,REM_skip_line_number)
# read nlj
nlj_ab = pd.read_csv(EM_path,
                 sep=r'\s+', 
                 skiprows=nlj_skip_line_number,
                 nrows=REM_skip_line_number-nlj_skip_line_number-4)
# a/b represents the same basis though ifg are different.
nlj_ab = nlj_ab.drop(columns=['ifg']).drop_duplicates().reset_index(drop=True)

# Convert 'j' column to float
def safe_fraction_convert(x):
    try:
        return pd.eval(x) if '/' in str(x) else float(x)
    except:
        return float(x)
nlj_ab['j'] = nlj_ab['j'].apply(safe_fraction_convert)


# read reduced matrix element
REM_ab = pd.read_csv(EM_path,
                 sep=r'\s+',
                 skiprows=REM_skip_line_number,
                 dtype=float)

# merge nlj and REM
df_a = nlj_ab.copy()
df_a = df_a.rename(columns={
    'a/b': 'a',
    'n': 'n_a',
    'l': 'l_a', 
    'j': 'j_a'
})

df_b = nlj_ab.copy()
df_b = df_b.rename(columns={
    'a/b': 'b',
    'n': 'n_b',
    'l': 'l_b', 
    'j': 'j_b'
})
merged_a = pd.merge(REM_ab, df_a, on=['a'], how='left')
REM = pd.merge(merged_a, df_b, on=['b'], how='left')
REM

3 71


,a,b,sqrt(4pi)<a||r^2Y_0||b>,<a||Q1||b>,<a||Q2||b>,n_a,l_a,j_a,n_b,l_b,j_b
0,1.0,1.0,5.497991,0.000000,0.000000,1,0,0.5,1,0,0.5
1,1.0,2.0,0.000000,-0.786601,0.000000,1,0,0.5,1,1,0.5
2,1.0,3.0,0.000000,-1.112422,0.000000,1,0,0.5,1,1,1.5
3,1.0,4.0,-0.000000,0.000000,2.831643,1,0,0.5,1,2,1.5
4,1.0,5.0,0.000000,0.000000,3.468040,1,0,0.5,1,2,2.5
...,...,...,...,...,...,...,...,...,...,...,...
1291,36.0,32.0,-0.000000,0.000000,-0.000000,4,1,1.5,2,5,5.5
1292,36.0,33.0,-0.000000,0.000000,-7.776899,4,1,1.5,3,3,2.5
1293,36.0,34.0,0.000000,0.000000,-19.049433,4,1,1.5,3,3,3.5
1294,36.0,35.0,0.000000,0.000000,-12.429160,4,1,1.5,4,1,0.5


## calculate kernel

### check Particle number
$$\hat{N} =  \sqrt{4\pi}\sum_{ab}\langle a||Y_{0}||b\rangle [c_a^\dagger\tilde{c}_b]_{00} = \sum_{ab} \delta_{ab}\sqrt{2j_a +1} \left[c^\dagger_a \tilde{c}_b\right]_{00}$$

So
$$ 
\bra{JMK\pi q} \hat{N} \ket{JMK'\pi'q'} =
 \sum_{ab}\delta_{ab} \sqrt{2j_a +1} \bra{JMK\pi q}\left[c^\dagger_a \tilde{c}_b\right]_{00}\ket{JMK'\pi' q'} 
$$
$$
 = \sum_{ab}\delta_{ab}\sqrt{2j_a +1} 
 \frac{1}{\sqrt{2J+1}}
 \bra{JK\pi q}|\left[c^\dagger_a \tilde{c}_b\right]_{0}|\ket{JK'\pi' q'} 
$$




In [34]:

for q1q2 in range(len(Res_data)):
    TDdens_path = os.path.join(output_dir,Res_data.iloc[q1q2]['TDdens_filename'])
    Kernel_path = os.path.join(output_dir,Res_data.iloc[q1q2]['Kernel_filename'])
    # read TD1B density
    TDdens = pd.read_csv(TDdens_path, sep=r'\s+')
    TDdens = TDdens.rename(columns={
        'l': 'lambda'
    })


    Ji = 0; Pi = '+'; Ki = 0
    Jf = Ji; Pf = '+'; Kf = Ki
    lambda_ = 0

    df_tmp = TDdens[(TDdens["Ji"]==Ji) &(TDdens["Ki"]==Ki) & (TDdens["Pi"]==Pi) & (TDdens["Jf"]==Jf) & (TDdens["Kf"]==Kf) & (TDdens["Pf"]==Pf)& (TDdens["lambda"]==lambda_) & (TDdens["a"]==TDdens["b"]) ]
    merged = pd.merge(df_tmp, REM, left_on=["a","b"], right_on=["a","b"], how="inner")
    merged["proton_weighted"]  = merged["proton"]  * (2*merged["j_a"]+1)**0.5
    merged["neutron_weighted"]  = merged["neutron"]  * (2*merged["j_a"]+1)**0.5
    proton_sum = merged["proton_weighted"].sum()/(2*Ji+1)**0.5
    neutron_sum  = merged["neutron_weighted"].sum()/(2*Ji+1)**0.5


    # read norm kernel
    with open(Kernel_path, 'r') as f:
        line = f.readlines()[Ji*4+1]
        Norm = float(line[:15].strip())
        Energy = float(line[30:45].strip())
        # print('Norm=',Norm)
    # print('J=',Ji)

    Res_data.loc[q1q2,f'Norm({Ji}{Pi})'] = Norm
    Res_data.loc[q1q2,f'Energy({Ji}{Pi})'] = Energy
    Res_data.loc[q1q2,f'N({Ji}{Pi})'] = neutron_sum/ Norm
    Res_data.loc[q1q2,f'Z({Ji}{Pi})'] = proton_sum/ Norm
Res_data

,beta2(qf),beta3(qf),beta2(qi),beta3(qi),TDdens_filename,Kernel_filename,Norm(0+),Energy(0+),N(0+),Z(0+)
0,0.0,0.0,0.0,0.0,Proj_TD1B.1D_eMax06.07.12+000+000_+000+000.me,Proj_kern.1D_eMax06.07.12+000+000_+000+000.elem,0.191072,-153.0213,10.0,10.0


### $E0$

$$\hat{T}(E0)  = e r^2 =  e\sqrt{4\pi}r^2 Y_{00} =  e\sqrt{4\pi}\sum_{ab}\langle a||r^2Y_{0}||b\rangle [c_a^\dagger\tilde{c}_b]_{00} $$

So, 
$$ 
\bra{JMK\pi q} \hat{T}(E0) \ket{JMK' \pi' q'} =
 e\sqrt{4\pi}
 \sum_{ab} \langle a||r^2Y_{0}||b\rangle \bra{JMK \pi q}\left[c^\dagger_a \tilde{c}_b\right]_{00}\ket{JMK'\pi' q'} 
 = e\sqrt{4\pi}\sum_{ab}\langle a||r^2Y_{0}||b\rangle 
    (-1)^{J-M}
    \left(\begin{array}{ccc}
        J & 0 & J'\\
        -M & 0 & M'
    \end{array}\right)
    \langle{J K \pi q}|| [c_a^\dagger\tilde{c}_b]_{0} ||{J' K' \pi' q'}\rangle\\
 = e\sqrt{4\pi}\sum_{ab}\langle a||r^2Y_{0}||b\rangle 
 \frac{1}{\sqrt{2J + 1}}
 \delta_{J J'} \delta_{M M'}    
\langle{J K \pi q}|| [c_a^\dagger\tilde{c}_b]_{0} ||{J' K' \pi' q'}\rangle\\
$$

In [ ]:
for q1q2 in range(len(Res_data)):
    TDdens_path = os.path.join(output_dir,Res_data.iloc[q1q2]['TDdens_filename'])
    Kernel_path = os.path.join(output_dir,Res_data.iloc[q1q2]['Kernel_filename'])
    # read TD1B density
    TDdens = pd.read_csv(TDdens_path, sep=r'\s+')
    TDdens = TDdens.rename(columns={
        'l': 'lambda'
    })

    Ji = 0; Pi = '+'; Ki = 0
    Jf = Ji; Pf = '+'; Kf = Ki
    lambda_ = 0
    
    df_tmp = TDdens[(TDdens["Ji"]==Ji) &(TDdens["Ki"]==Ki) & (TDdens["Pi"]==Pi) & (TDdens["Jf"]==Jf) & (TDdens["Kf"]==Kf) & (TDdens["Pf"]==Pf)& (TDdens["lambda"]==lambda_)]
    merged = pd.merge(df_tmp, REM, left_on=["a","b"], right_on=["a","b"], how="inner")
    merged["neutron_weighted"] = merged["neutron"] * merged["sqrt(4pi)<a||r^2Y_0||b>"]
    merged["proton_weighted"]  = merged["proton"]  * merged["sqrt(4pi)<a||r^2Y_0||b>"]
    neutron_sum = merged["neutron_weighted"].sum()
    proton_sum  = merged["proton_weighted"].sum()

    # Res_data.loc[q1q2,f'<{Jf}{Pf}||E{lambda_}||{Ji}{Pi}>(neutron)'] = neutron_sum/math.sqrt(2*Ji+1)
    Res_data.loc[q1q2,f'<{Jf}{Pf}||E{lambda_}||{Ji}{Pi}>(proton)'] = proton_sum/math.sqrt(2*Ji+1)
Res_data


,beta2(qf),beta3(qf),beta2(qi),beta3(qi),TDdens_filename,Kernel_filename,Norm(0+),Energy(0+),N(0+),Z(0+),<0+||E0||0+>(neutron),<0+||E0||0+>(proton)
0,0.0,0.0,0.0,0.0,Proj_TD1B.1D_eMax06.07.12+000+000_+000+000.me,Proj_kern.1D_eMax06.07.12+000+000_+000+000.elem,0.191072,-153.0213,10.0,10.0,14.690413,14.950783


### $E\lambda$ reduced matrix element
$$\hat{Q}_{\lambda\mu} = \hat{\lambda}^{-1} \sum_{ab} \bra{a}|\hat{Q}_\lambda|\ket{b} \left[c^\dagger_a \tilde{c}_b\right]_{\lambda\mu}$$
So,
$$ 
\bra{J_fK_f q_f \pi_f}| \hat{Q}_\lambda |\ket{J_i K_i q_i \pi_i} =
 \lambda^{-1} \sum_{ab} \bra{a}|\hat{Q}_\lambda|\ket{b} \bra{J_fK_f q_f \pi_f}| \left[c^\dagger_a \tilde{c}_b\right]_{\lambda} |\ket{J_i K_i q_i \pi_i} 
$$


In [26]:
for q1q2 in range(len(Res_data)):
    TDdens_path = os.path.join(output_dir,Res_data.iloc[q1q2]['TDdens_filename'])
    Kernel_path = os.path.join(output_dir,Res_data.iloc[q1q2]['Kernel_filename'])
    # read TD1B density
    TDdens = pd.read_csv(TDdens_path, sep=r'\s+')
    TDdens = TDdens.rename(columns={
        'l': 'lambda'
    })

    # E2(0+ -> 2+)
    Ji = 0; Pi = '+'; Ki = 0
    Jf = 2; Pf = '+'; Kf = 0
    lambda_ = 2

    df_tmp = TDdens[(TDdens["Ji"]==Ji) &(TDdens["Ki"]==Ki) & (TDdens["Pi"]==Pi) & (TDdens["Jf"]==Jf) & (TDdens["Kf"]==Kf) & (TDdens["Pf"]==Pf)& (TDdens["lambda"]==lambda_)]
    merged = pd.merge(df_tmp, REM, left_on=["a","b"], right_on=["a","b"], how="inner")
    merged["neutron_weighted"] = merged["neutron"] * merged[f"<a||Q{lambda_}||b>"]
    merged["proton_weighted"]  = merged["proton"]  * merged[f"<a||Q{lambda_}||b>"]
    neutron_sum = merged["neutron_weighted"].sum()
    proton_sum  = merged["proton_weighted"].sum()

    Res_data.loc[q1q2,f'<{Jf}{Pf}||E{lambda_}||{Ji}{Pi}>(neutron)'] = neutron_sum/math.sqrt(2*lambda_+1)
    Res_data.loc[q1q2,f'<{Jf}{Pf}||E{lambda_}||{Ji}{Pi}>(proton)'] = proton_sum/math.sqrt(2*lambda_+1)

    # E2(2+ -> 4+)
    Ji = 2; Pi = '+'; Ki = 0
    Jf = 4; Pf = '+'; Kf = 0
    lambda_ = 2

    df_tmp = TDdens[(TDdens["Ji"]==Ji) &(TDdens["Ki"]==Ki) & (TDdens["Pi"]==Pi) & (TDdens["Jf"]==Jf) & (TDdens["Kf"]==Kf) & (TDdens["Pf"]==Pf)& (TDdens["lambda"]==lambda_)]
    merged = pd.merge(df_tmp, REM, left_on=["a","b"], right_on=["a","b"], how="inner")
    merged["neutron_weighted"] = merged["neutron"] * merged[f"<a||Q{lambda_}||b>"]
    merged["proton_weighted"]  = merged["proton"]  * merged[f"<a||Q{lambda_}||b>"]
    neutron_sum = merged["neutron_weighted"].sum()
    proton_sum  = merged["proton_weighted"].sum()

    Res_data.loc[q1q2,f'<{Jf}{Pf}||E{lambda_}||{Ji}{Pi}>(neutron)'] = neutron_sum/math.sqrt(2*lambda_+1)
    Res_data.loc[q1q2,f'<{Jf}{Pf}||E{lambda_}||{Ji}{Pi}>(proton)'] = proton_sum/math.sqrt(2*lambda_+1)

    # # E3(2+ -> 3-)
    # Ji = 2; Pi = '+'; Ki = 0
    # Jf = 3; Pf = '-'; Kf = 0
    # lambda_ = 3

    # df_tmp = TDdens[(TDdens["Ji"]==Ji) &(TDdens["Ki"]==Ki) & (TDdens["Pi"]==Pi) & (TDdens["Jf"]==Jf) & (TDdens["Kf"]==Kf) & (TDdens["Pf"]==Pf)& (TDdens["lambda"]==lambda_)]
    # merged = pd.merge(df_tmp, REM, left_on=["a","b"], right_on=["a","b"], how="inner")
    # merged["neutron_weighted"] = merged["neutron"] * merged[f"<a||Q{lambda_}||b>"]
    # merged["proton_weighted"]  = merged["proton"]  * merged[f"<a||Q{lambda_}||b>"]
    # neutron_sum = merged["neutron_weighted"].sum()
    # proton_sum  = merged["proton_weighted"].sum()

    # Res_data.loc[q1q2,f'<{Jf}{Pf}||E{lambda_}||{Ji}{Pi}>(neutron)'] = neutron_sum/math.sqrt(2*lambda_+1)
    # Res_data.loc[q1q2,f'<{Jf}{Pf}||E{lambda_}||{Ji}{Pi}>(proton)'] = proton_sum/math.sqrt(2*lambda_+1)

Res_data

,beta2(qf),beta3(qf),beta2(qi),beta3(qi),TDdens_filename,Kernel_filename,Norm(0+),Energy(0+),N(0+),Z(0+),<0+||E0||0+>(proton),<2+||E2||0+>(neutron),<2+||E2||0+>(proton),<4+||E2||2+>(neutron),<4+||E2||2+>(proton)
0,0.0,0.0,0.0,0.0,Proj_TD1B.1D_eMax06.07.12+000+000_+000+000.me,Proj_kern.1D_eMax06.07.12+000+000_+000+000.elem,0.191072,-153.0213,10.0,10.0,14.950783,3.025326e-07,-1.019841e-07,-5.360931e-11,-5.076832e-11


In [27]:
Res_data.drop(columns=['TDdens_filename', 'Kernel_filename'], inplace=True)
Res_data

,beta2(qf),beta3(qf),beta2(qi),beta3(qi),Norm(0+),Energy(0+),N(0+),Z(0+),<0+||E0||0+>(proton),<2+||E2||0+>(neutron),<2+||E2||0+>(proton),<4+||E2||2+>(neutron),<4+||E2||2+>(proton)
0,0.0,0.0,0.0,0.0,0.191072,-153.0213,10.0,10.0,14.950783,3.025326e-07,-1.019841e-07,-5.360931e-11,-5.076832e-11
